In [8]:
import pandas as pd
import numpy as np
import os

In [10]:
os.getcwd()
# 'C:\\Users\\gcart'

'C:\\Users\\gcart'

In [11]:
os.chdir('E:/docs/book_reviews/manning/solve_any_data_analysis_problem/notebooks')
os.getcwd()

'E:\\docs\\book_reviews\\manning\\solve_any_data_analysis_problem\\notebooks'

In [13]:
sales = pd.read_csv("purchases.csv")
print(sales.shape)

(71519, 11)


In [2]:
sales.isnull().sum()

event_time              0
product_id              0
category_id             0
category_code       16739
brand                5707
price                   0
session_id              0
customer_id         18448
guest_first_name    53071
guest_surname       53071
guest_postcode      53071
dtype: int64

In [ ]:
# 18448 (missing Cust_ids) + 53071 (mssing guest data, cause we have actual cust data) = 71519 (total records)

In [15]:
# create a new column to track guest checkouts, which happen when a customer ID is not provided.

sales["is_guest"] = sales["customer_id"].isnull()

In [16]:
# code checks for cases where the transaction was a guest checkout, but we also had a customer ID, and the second returns cases where we had neither.

sales[sales["is_guest"] & sales["customer_id"].notnull()]


,event_time,product_id,category_id,category_code,brand,price,session_id,customer_id,guest_first_name,guest_surname,guest_postcode,is_guest


In [17]:
sales[(sales["is_guest"] == False) & sales["customer_id"].isnull()]

,event_time,product_id,category_id,category_code,brand,price,session_id,customer_id,guest_first_name,guest_surname,guest_postcode,is_guest


In [18]:
# find distribution of 'guest' customers

sales["is_guest"].value_counts(normalize=True)

False    0.742055
True     0.257945
Name: is_guest, dtype: float64

In [19]:
# 'guest' may be duplicates, find uniques, then recalculate 'guest' distribution

guest_columns = ["guest_first_name", "guest_surname", "guest_postcode"]

# get a unique combination of guest columns
unique_guests = sales[guest_columns].drop_duplicates()
print(len(unique_guests))

# get all unique customer IDs
unique_customers = sales["customer_id"].unique()
cust_total = len(unique_customers) + len(unique_guests)

# subtract 1 from the unique customer count because NULL is also counted
print(len(unique_guests) / (cust_total-1))

8301
0.2495640671036017


In [20]:
# export only the guests, we can filter using our is_guest column and export only the relevant columns.

guest_columns = ["guest_first_name", "guest_surname", "guest_postcode", "is_guest"]
guests = sales.loc[sales["is_guest"], guest_columns]
guests = guests.drop_duplicates()
guests.head()

,guest_first_name,guest_surname,guest_postcode,is_guest
0,MICHAEL,MASON,RG497ZQ,True
2,COLE,WILKINSON,SW75TQ,True
3,MOHAMMED,RICHARDS,RG150RE,True
7,KIAN,MILLS,SW332TF,True
13,RUBY,OWEN,PO377YS,True


In [22]:
# For non-guest checkouts, we won’t have these columns; we will have only a customer ID:

non_guests = (
    
  # customer ID is a single column, so we need to explicitly make it a DataFrame
  pd.DataFrame(
    sales.loc[sales["customer_id"].notnull(), "customer_id"]
      # we extract unique customer IDs from non-guest rows
      .unique()
      .astype(int),
    columns=["customer_id"]
  )
)
non_guests.head()

,customer_id
0,7466
1,31266
2,534142828
3,1035
4,6985


In [24]:
# ll the columns from both datasets and missing data where a column did not exist in one of the datasets.

sales_customers = pd.concat([non_guests,guests], axis=0, ignore_index=True)

In [25]:
# rename our columns and remove the guest prefix.

new_col_names = ["customer_id", "first_name", "surname", "postcode", "is_guest"]
sales_customers = sales_customers.set_axis(new_col_names, axis=1)

In [26]:
# fill in the missing values for the is_guest column. 

sales_customers["is_guest"] = sales_customers["is_guest"].fillna(False)

In [27]:
# add the in_purchase_data column we decided on for our schema

sales_customers["in_purchase_data"] = True